# RAG - self_train_data + LORA 
自检索 + 微调


lora  
LORA_R = 8                
LORA_ALPHA = 16           
LORA_DROPOUT = 0.1         
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"]  
BIAS = "none"

训练参数  
NUM_TRAIN_EPOCHS = 4 #5  
BATCH_SIZE = 4  
GRADIENT_ACCUMULATION_STEPS = 8  
LEARNING_RATE = 2e-5  
MAX_GRAD_NORM = 1.0  
SAVE_STEPS = 500  
LOGGING_STEPS = 50  
SEED = 42  
<!-- MAX_LENGTH = 512 -->
MAX_LENGTH = 1024  
WEIGHT_DECAY = 0.01  
LR_SCHEDULER_TYPE = "cosine"        # 余弦退火  
EARLY_STOPPING_PATIENCE = 3



In [1]:
import os
import sys
from pathlib import Path
import json
current_dir = Path.cwd() 
project_root = os.path.dirname(os.path.dirname(current_dir))     # 项目根目录
sys.path.insert(0, project_root)
from config.config import SYSTEM_MESSAGE, INSTRUCTION

data_output_dir = os.path.join(project_root, "data")
train_data_path = os.path.join(data_output_dir, "rag_self_train.json")
test_data_path = os.path.join(data_output_dir, "rag_self_test.json")
data_path = os.path.join(data_output_dir, "output/lora_results_self_train.json")
print(data_path,test_data_path)


/mnt/workspace/Graduate/ToxiCN_lora_rag/data/output/lora_results_self_train.json /mnt/workspace/Graduate/ToxiCN_lora_rag/data/rag_self_test.json


In [2]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train = load_json(train_data_path)
test = load_json(test_data_path)
print(f"训练数据长度：{len(train)}")
print(f"训练数据长度：{len(test)}")


训练数据长度：6424
训练数据长度：1605


In [3]:
from src.fine_Tuning.train import train_main
lora_train_path = train_data_path
lora_save_path = "lora_self_train_adapter"
train_main(lora_train_path, lora_save_path)

/root/miniconda3/envs/llmtoxi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen3-1.7B LoRA微调脚本
设备: NVIDIA A10
PyTorch版本: 2.10.0+cu128
CUDA可用: True

[步骤2/5] 加载模型和分词器...
开始加载模型和分词器...


`torch_dtype` is deprecated! Use `dtype` instead!


加载基础模型...


Loading checkpoint shards: 100%|██████████| 4/4 [00:33<00:00,  8.45s/it]


...


[步骤4/5] 训练LoRA适配器...
开始配置训练参数...
{'input_ids': [151644, 8948, 198, 56568, 101909, 43815, 104998, 101057, 3837, 107618, 101042, 104811, 117828, 109445, 11, 107618, 45181, 108704, 15946, 107439, 55338, 117828, 109445, 63703, 23305, 40027, 1773, 151645, 198, 151644, 872, 198, 14880, 45181, 100431, 9370, 108704, 15946, 118797, 46944, 57191, 101213, 63703, 23305, 40027, 8997, 103991, 63703, 23305, 40027, 9370, 68805, 17714, 5122, 85641, 64429, 91, 64429, 101313, 91, 117828, 104553, 91, 64471, 117828, 198, 100470, 101882, 28311, 220, 220, 16, 13, 220, 100783, 108704, 64471, 102298, 117828, 109445, 3837, 100645, 104102, 66017, 46944, 63703, 23305, 40027, 3837, 53153, 91680, 66017, 508, 4689, 60, 92313, 102158, 104136, 33071, 87335, 8997, 220, 220, 17, 13, 49602, 249, 23305, 40027, 15946, 100802, 44931, 71268, 100645, 100347, 3837, 11622, 105205, 107899, 43268, 760, 58657, 99859, 3837, 53153, 102822, 108112, 105060, 3837, 91572, 100148, 37029, 71134, 22243, 109619, 8997, 220, 220, 18, 13, 812

The model is already on multiple devices. Skipping the move to device specified in `args`.


验证集总样本数: 1285
有效 token 为 0 的样本数: 0
开始训练...


Step,Training Loss,Validation Loss
100,0.309600,0.278677
200,0.231700,0.241223
300,0.227400,0.226269
400,0.192900,0.220251
500,0.205100,0.216281
600,0.186400,0.215707


保存LoRA适配器...
训练完成！模型保存在: /mnt/workspace/Graduate/ToxiCN_lora_rag/output/lora_self_train_adapter/train


In [3]:
from src.fine_Tuning.inference import inference_main
lora_model_path = os.path.join(project_root, "output", 'lora_self_train_adapter/train')
inference_main(test_data_path, lora_model_path, data_path)

/root/miniconda3/envs/llmtoxi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


加载测试数据...
初始化推理引擎...
INFO 04-09 15:31:24 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'disable_log_stats': True, 'enable_lora': True, 'model': '/mnt/workspace/Graduate/ToxiCN_lora_rag/model/Qwen2.5-7B-Instruct'}
INFO 04-09 15:31:24 [model.py:533] Resolved architecture: Qwen2ForCausalLM
INFO 04-09 15:31:24 [model.py:1582] Using max model len 2048
INFO 04-09 15:31:24 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-09 15:31:24 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=42299) INFO 04-09 15:31:25 [core.py:103] Initializing a V1 LLM engine (v0.18.1) with config: model='/mnt/workspace/Graduate/ToxiCN_lora_rag/model/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='/mnt/workspace/Graduate/ToxiCN_lora_rag/model/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, dow

(EngineCore pid=42299) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=42299) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:21<01:03, 21.07s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:42<00:42, 21.43s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [01:04<00:21, 21.55s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:04<00:00, 13.26s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:04<00:00, 16.25s/it]
(EngineCore pid=42299) 


(EngineCore pid=42299) INFO 04-09 15:32:33 [default_loader.py:384] Loading weights took 65.16 seconds
(EngineCore pid=42299) INFO 04-09 15:32:33 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=42299) INFO 04-09 15:32:33 [gpu_model_runner.py:4566] Model loading took 14.32 GiB memory and 65.898342 seconds
(EngineCore pid=42299) INFO 04-09 15:32:37 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/79a954f42d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=42299) INFO 04-09 15:32:37 [backends.py:1048] Dynamo bytecode transform time: 3.55 s
(EngineCore pid=42299) INFO 04-09 15:32:40 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.385 s
(EngineCore pid=42299) INFO 04-09 15:32:40 [monitor.py:48] torch.compile took 6.39 s in total
(EngineCore pid=42299) INFO 04-09 15:32:40 [decorators.py:296] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_com

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:10<00:00, 10.04it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 70/70 [00:05<00:00, 12.28it/s]


(EngineCore pid=42299) INFO 04-09 15:33:01 [gpu_model_runner.py:5746] Graph capturing finished in 17 secs, took 1.16 GiB
(EngineCore pid=42299) INFO 04-09 15:33:01 [gpu_worker.py:617] CUDA graph pool memory: 1.16 GiB (actual), 0.92 GiB (estimated), difference: 0.24 GiB (20.5%).
(EngineCore pid=42299) INFO 04-09 15:33:01 [core.py:281] init engine (profile, create kv cache, warmup model) took 27.46 seconds
INFO 04-09 15:33:02 [llm.py:391] Supported tasks: ['generate']
提取所有问题...
批量推理...


批量推理:   0%|          | 0/201 [00:00<?, ?it/s]

WARNING 04-09 15:33:02 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


批量推理: 100%|██████████| 201/201 [06:06<00:00,  1.82s/it]


(EngineCore pid=42299) INFO 04-09 15:39:08 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=42299) INFO 04-09 15:39:08 [core.py:1224] Shutdown complete
保存结果...
结果保存至 /mnt/workspace/Graduate/ToxiCN_lora_rag/data/output/lora_results_self_train.json


In [2]:
from src.eval import main
main(data_path)

四元组硬匹配结果： (0.18396711202466598, 0.1882229232386961, 0.18607068607068605)
四元组软匹配结果： (0.37358684480986637, 0.3822292323869611, 0.37785862785862784)
三元组硬匹配结果： (0.21891058581706063, 0.22397476340694006, 0.2214137214137214)
三元组软匹配结果： (0.45940390544707094, 0.47003154574132494, 0.46465696465696466)
二元组硬匹配结果： (0.2687564234326824, 0.27497371188222924, 0.27182952182952186)
二元组软匹配结果： (0.5626927029804728, 0.5757097791798107, 0.5691268191268192)
单元素硬匹配结果： (0.32476875642343267, 0.3322818086225026, 0.32848232848232845)
单元素软匹配结果： (0.6690647482014388, 0.6845425867507886, 0.6767151767151766)


{'quadruple_hard_match': (0.18396711202466598,
  0.1882229232386961,
  0.18607068607068605),
 'quadruple_soft_match': (0.37358684480986637,
  0.3822292323869611,
  0.37785862785862784),
 'triple_hard_match': (0.21891058581706063,
  0.22397476340694006,
  0.2214137214137214),
 'triple_soft_match': (0.45940390544707094,
  0.47003154574132494,
  0.46465696465696466),
 'pair_hard_match': (0.2687564234326824,
  0.27497371188222924,
  0.27182952182952186),
 'pair_soft_match': (0.5626927029804728,
  0.5757097791798107,
  0.5691268191268192),
 'single_element_hard_match': (0.32476875642343267,
  0.3322818086225026,
  0.32848232848232845),
 'single_element_soft_match': (0.6690647482014388,
  0.6845425867507886,
  0.6767151767151766)}

# RAG-Lexicon + Lora

In [ ]:
import os
import sys
from pathlib import Path
import json
current_dir = Path.cwd() 
project_root = os.path.dirname(os.path.dirname(current_dir))     # 项目根目录
sys.path.insert(0, project_root)
from config.config import SYSTEM_MESSAGE, INSTRUCTION

data_output_dir = os.path.join(project_root, "data")
train_data_path = os.path.join(data_output_dir, "lexicon_rag_train_data.json")
test_data_path = os.path.join(data_output_dir, "lexicon_rag_test_data.json")
save_data_path = os.path.join(data_output_dir, "output/lora_results_lexicon.json")
print(data_path,test_data_path)

In [ ]:
from src.fine_Tuning.train import train_main
lora_train_path = train_data_path
lora_save_path = "lora_lexicon_apdater"
train_main(lora_train_path, lora_save_path)

In [ ]:
from src.fine_Tuning.inference import inference_main
lora_model_path = os.path.join(project_root, "output", lora_save_path)
inference_main(test_data_path, lora_model_path, save_data_path)

# RAG - self_train
自检索+基础模型

In [ ]:
from src.RAG.inference import run_inference
test_data_path = os.path.join(data_output_dir, "rag_self_test.json")
save_data_path = os.path.join(data_output_dir, "output", "result_base_self_train.json")
run_inference(test_data_path, save_data_path, batch_size=8)

In [4]:
from src.eval import main
main(save_data_path)

四元组硬匹配结果： (0.041187384044526903, 0.0583596214511041, 0.0482923645855993)
四元组软匹配结果： (0.13803339517625232, 0.19558359621451105, 0.16184468131390034)
三元组硬匹配结果： (0.0679035250463822, 0.09621451104100946, 0.07961714161409615)
三元组软匹配结果： (0.21743970315398886, 0.30809674027339645, 0.2549488797041549)
二元组硬匹配结果： (0.09165120593692022, 0.129863301787592, 0.1074613878616489)
二元组软匹配结果： (0.2964749536178108, 0.42008412197686645, 0.3476180117467914)
单元素硬匹配结果： (0.1365491651205937, 0.19348054679284962, 0.16010441592342833)
单元素软匹配结果： (0.40630797773654914, 0.5757097791798107, 0.47639765064172285)


{'quadruple_hard_match': (0.041187384044526903,
  0.0583596214511041,
  0.0482923645855993),
 'quadruple_soft_match': (0.13803339517625232,
  0.19558359621451105,
  0.16184468131390034),
 'triple_hard_match': (0.0679035250463822,
  0.09621451104100946,
  0.07961714161409615),
 'triple_soft_match': (0.21743970315398886,
  0.30809674027339645,
  0.2549488797041549),
 'pair_hard_match': (0.09165120593692022,
  0.129863301787592,
  0.1074613878616489),
 'pair_soft_match': (0.2964749536178108,
  0.42008412197686645,
  0.3476180117467914),
 'single_element_hard_match': (0.1365491651205937,
  0.19348054679284962,
  0.16010441592342833),
 'single_element_soft_match': (0.40630797773654914,
  0.5757097791798107,
  0.47639765064172285)}

# RAG - lexicon
俚语词表+基础模型

In [ ]:

from src.RAG.inference import run_inference
test_data_path = os.path.join(data_output_dir, "lexicon_rag_test_data.json")
save_data_path = os.path.join(data_output_dir, "output", "result_base_lexicon.json")
run_inference(test_data_path,save_data_path , batch_size=8)

In [3]:
from src.eval import main
main(save_data_path)

四元组硬匹配结果： (0.029411764705882353, 0.050473186119873815, 0.03716608594657375)
四元组软匹配结果： (0.09283088235294118, 0.15930599369085174, 0.11730545876887341)
三元组硬匹配结果： (0.050551470588235295, 0.08675078864353312, 0.06387921022067364)
三元组软匹配结果： (0.15839460784313725, 0.2718191377497371, 0.20015485869144403)
二元组硬匹配结果： (0.07322303921568628, 0.1256572029442692, 0.09252806813782423)
二元组软匹配结果： (0.22181372549019607, 0.38065194532071506, 0.2802942315137437)
单元素硬匹配结果： (0.11856617647058823, 0.20347003154574134, 0.14982578397212543)
单元素软匹配结果： (0.359375, 0.6167192429022083, 0.45412311265969807)


{'quadruple_hard_match': (0.029411764705882353,
  0.050473186119873815,
  0.03716608594657375),
 'quadruple_soft_match': (0.09283088235294118,
  0.15930599369085174,
  0.11730545876887341),
 'triple_hard_match': (0.050551470588235295,
  0.08675078864353312,
  0.06387921022067364),
 'triple_soft_match': (0.15839460784313725,
  0.2718191377497371,
  0.20015485869144403),
 'pair_hard_match': (0.07322303921568628,
  0.1256572029442692,
  0.09252806813782423),
 'pair_soft_match': (0.22181372549019607,
  0.38065194532071506,
  0.2802942315137437),
 'single_element_hard_match': (0.11856617647058823,
  0.20347003154574134,
  0.14982578397212543),
 'single_element_soft_match': (0.359375,
  0.6167192429022083,
  0.45412311265969807)}